<a href="https://colab.research.google.com/github/toddbalwinski/ds2002-fa26/blob/main/2026-09-16%20%E2%80%94%20Pandas%20Core%20Ops%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Core Ops

**Studio — 2026-09-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Building a vendor report

Today you build one deliverable end to end: a per-vendor summary that a game-day manager could act on. Three sources, a join that does not behave, and a report at the end.

The data has problems planted in it. Finding them is part of the work — a report you cannot defend is worth nothing, however good the code looks.

In [1]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,item,qty,price
1,V-01,Cheeseburger,2,7.50
2,V-10,Foam Finger,1,12.00
3,V-01,Hot Dog,3,4.50
4,V-18,Rain Poncho,5,6.00
5,V-10,UVA T-Shirt,1,24.00
6,V-05,Chicken Tacos,4,6.50
7,V-18,Rain Poncho,8,6.00
8,V-42,Kettle Corn,3,5.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C
V-18,Rally Rain Gear,C'''))

targets = pd.read_csv(StringIO('''zone,revenue_target
A,80
B,25
C,60'''))

print('orders:', orders.shape, '| vendors:', vendors.shape, '| targets:', targets.shape)
orders

orders: (8, 5) | vendors: (5, 3) | targets: (3, 2)


,order_id,vendor_id,item,qty,price
0,1,V-01,Cheeseburger,2,7.5
1,2,V-10,Foam Finger,1,12.0
2,3,V-01,Hot Dog,3,4.5
3,4,V-18,Rain Poncho,5,6.0
4,5,V-10,UVA T-Shirt,1,24.0
5,6,V-05,Chicken Tacos,4,6.5
6,7,V-18,Rain Poncho,8,6.0
7,8,V-42,Kettle Corn,3,5.0


### Worked example — the merge, done carefully

Here is one merge done properly, so the pattern is on the screen before you write anything. Three things happen: record the baseline, merge with `indicator=True`, then compare against the baseline.

In [2]:
baseline_rows = len(orders)
orders['revenue'] = orders['qty'] * orders['price']
baseline_revenue = orders['revenue'].sum()
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f}')

check = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print(f'after:  {len(check)} rows, ${check["revenue"].sum():.2f}')
print()
print(check['_merge'].value_counts())

before: 8 rows, $183.50
after:  10 rows, $261.50

_merge
both          9
left_only     1
right_only    0
Name: count, dtype: int64


Eight orders went in and nine came out, and the revenue total moved. Both symptoms point at the same cause, and it is in the `vendors` table, not in the orders.

Find it before you go further — everything downstream inherits this bug.

In [3]:
# Which vendor_id appears more than once in the vendor list?
print(vendors['vendor_id'].value_counts())
print()
print('duplicated vendor rows:', vendors.duplicated().sum())

vendor_id
V-18    2
V-01    1
V-05    1
V-10    1
Name: count, dtype: int64

duplicated vendor rows: 1


### Build 1 — fix the vendor list, then merge

**TODO:** drop the duplicate vendor row, then join it onto `orders` with an indicator. Your merge must come out at **8 rows** with the revenue total unchanged from the baseline. Print both to prove it.

In [5]:
# TODO: clean_vendors = ...
# TODO: joined = orders.merge(...)
# TODO: print row count and revenue, and compare to baseline_rows / baseline_revenue

clean_vendors = vendors.drop_duplicates(subset='vendor_id')

joined = orders.merge(
    clean_vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

joined['revenue'] = joined['qty'] * joined['price']

print("Row count:", len(joined))
print("Baseline rows:", baseline_rows)

print("Revenue:", joined['revenue'].sum())
print("Baseline revenue:", baseline_revenue)

Row count: 8
Baseline rows: 8
Revenue: 183.5
Baseline revenue: 183.5


### Build 2 — handle the vendor nobody has heard of

One order belongs to a vendor that is not on the roster. You have three options, and this is a judgment call:

1. Drop it — clean report, understated revenue.
2. Keep it with a blank name — honest, but it will show up as `NaN` in every chart.
3. Label it `'Unknown vendor'` and keep it in a zone called `'Unassigned'`.

**TODO:** pick one, implement it, and write one sentence saying why. Print how much revenue the decision affects either way.

In [6]:
# TODO: implement your choice
# TODO: print the revenue attached to the unmatched order

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined['zone'] = joined['zone'].fillna('Unassigned')

# Find revenue attached to the unmatched order
unmatched_revenue = joined.loc[
    joined['_merge'] == 'left_only',
    'revenue'
].sum()

print("Revenue from unmatched vendor:", unmatched_revenue)

Revenue from unmatched vendor: 15.0


**My decision, and why:** I chose to label it as an unknown vendor because we collected the money and made a transaction, we just dont know who the vendor was. If we drop the data entirely we could have mismatched earnings or could forget about the problem before it is solved.

### Build 3 — the per-vendor summary

**TODO:** one row per vendor, with:

- `orders` — how many orders
- `units` — total quantity
- `revenue` — total revenue
- `avg_ticket` — average revenue per order, rounded to 2 decimals

Sorted by revenue, highest first. Use `.agg()` with named outputs so the columns come out with the names above.

In [9]:
# TODO

vendor_summary = (
    joined
    .groupby(['vendor_id', 'vendor_name', 'zone'])
    .agg(
        orders=('order_id', 'count'),
        units=('qty', 'sum'),
        revenue=('revenue', 'sum'),
        avg_ticket=('revenue', 'mean')
    )
    .reset_index()
)

vendor_summary['avg_ticket'] = vendor_summary['avg_ticket'].round(2)

vendor_summary = vendor_summary.sort_values('revenue', ascending= False)

vendor_summary

,vendor_id,vendor_name,zone,orders,units,revenue,avg_ticket
3,V-18,Rally Rain Gear,C,2,13,78.0,39.00
2,V-10,Cav Merch North,A,2,2,36.0,18.00
0,V-01,Hoos Burgers,A,2,5,28.5,14.25
1,V-05,Rotunda Tacos,B,1,4,26.0,26.00
4,V-42,Unknown vendor,Unassigned,1,3,15.0,15.00


### Build 4 — did each zone hit its target?

**TODO:** total revenue by zone, join `targets` on, and add a `hit_target` boolean column. Then print a one-line sentence for each zone that a manager could read.

In [15]:
# TODO

zone_summary = (
    joined
    .groupby('zone', as_index=False)
    .agg(revenue=('revenue', 'sum'))
)

zone_summary = zone_summary.merge(
    targets,
    on='zone',
    how='left'
)

zone_summary['hit_target'] = (
    zone_summary['revenue'] >= zone_summary['revenue_target']
)

zone_summary

for _, row in zone_summary.iterrows():
    if pd.isna(row['revenue_target']):
        print(f"Zone {row['zone']} generated ${row['revenue']:.2f} in revenue but does not have an assigned target.")
    elif row['hit_target']:
        print(f"Zone {row['zone']} hit its target with ${row['revenue']:.2f} in revenue against a ${row['revenue_target']:.2f} target.")
    else:
        print(f"Zone {row['zone']} missed its target with ${row['revenue']:.2f} in revenue against a ${row['revenue_target']:.2f} target.")

Zone A missed its target with $64.50 in revenue against a $80.00 target.
Zone B hit its target with $26.00 in revenue against a $25.00 target.
Zone C hit its target with $78.00 in revenue against a $60.00 target.
Zone Unassigned generated $15.00 in revenue but does not have an assigned target.


### Build 5 — the one number that matters

**TODO:** rain gear is the thing we can actually act on. Print total poncho units sold and what share of overall revenue they represent, as a percentage rounded to one decimal.

In [16]:
# TODO

ponchos = joined[joined['item'] == 'Rain Poncho']

poncho_units = ponchos['qty'].sum()
poncho_revenue = ponchos['revenue'].sum()
overall_revenue = joined['revenue'].sum()

poncho_share = (poncho_revenue / overall_revenue) * 100

print("Total poncho units sold:", poncho_units)
print(f"Ponchos represent {poncho_share:.1f}% of overall revenue.")

Total poncho units sold: 13
Ponchos represent 42.5% of overall revenue.


---

## Checkpoint (participation)

Report your row count and revenue total after the merge, and what you decided to do with the unknown vendor.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [17]:
# Checkpoint
rows_after_merge = 8    # TODO: should equal 8
revenue_after_merge = 183.5 # TODO: should equal the baseline
unknown_vendor_call = 'I made it unknown so that it would show in reports and can be discussed and reviewed later'  # TODO: what you did with V-42, and why

print('rows after merge:', rows_after_merge)
print('revenue after merge:', revenue_after_merge)
print('unknown vendor:', unknown_vendor_call)

rows after merge: 8
revenue after merge: 183.5
unknown vendor: I made it unknown so that it would show in reports and can be discussed and reviewed later
